# **FastAPI Assignment 4 – Shopping Cart System**
### Objective
- The objective of this assignment is to develop a robust, stateful Shopping Cart API using FastAPI. It focuses on complex business logic, such as calculating subtotals, managing item quantities, validating stock availability, and processing checkouts into a permanent order history.

### Technologies Used
- Python: Core programming language.
- FastAPI: Modern, high-performance web framework for building APIs.
- Pydantic: Data validation and settings management using Python type annotations.
- Uvicorn: Lightning-fast ASGI server implementation.
- Swagger UI: Interactive API documentation for real-time testing.

# **Question 1: Add Items to the Cart**
- In a shopping cart system, the POST method is used to add products to a user's session. Each time a product is added, the system must fetch its details (like name and unit price) from the inventory and calculate the subtotal based on the quantity requested.
- This ensures that the cart reflects the correct price for each individual line item.

In [44]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel
from typing import List, Dict

app = FastAPI()

# Initial Inventory Data
inventory = {
    1: {"name": "Wireless Mouse", "price": 499, "in_stock": True},
    2: {"name": "Notebook", "price": 99, "in_stock": True},
    3: {"name": "USB Hub", "price": 799, "in_stock": False},
    4: {"name": "Pen Set", "price": 49, "in_stock": True}
}

# In-memory storage for Cart and Orders
cart = []
orders = []

@app.post("/cart/add")
def add_to_cart(product_id: int, quantity: int = 1):
    # Check if product exists in inventory
    if product_id not in inventory:
        raise HTTPException(status_code=404, detail="Product not found")
    
    product = inventory[product_id]
    
    # Logic to create a new cart item entry
    new_item = {
        "product_id": product_id,
        "product_name": product["name"],
        "quantity": quantity,
        "unit_price": product["price"],
        "subtotal": product["price"] * quantity
    }
    
    # Temporary logic for Q1 (We will add duplicate check in Q4)
    cart.append(new_item)
    return {"message": "Added to cart", "cart_item": new_item}

# **Question 2: View the Cart and Verify the Total**
- Once items are added to the cart, the system must provide a way to review the selection.
- The GET /cart endpoint is responsible for aggregating all the items currently in the session.
- A critical part of this process is calculating the Grand Total, which is the sum of all individual item subtotals.
- This allows the user to verify their total expenditure before proceeding to payment.

In [68]:
@app.get("/cart")
def view_cart():
    # Check if the cart is empty to return a clean response
    if not cart:
        return {
            "message": "Cart is empty", 
            "items": [], 
            "item_count": 0, 
            "grand_total": 0
        }
    
    # Task: Calculate the grand total manually from subtotals
    # Calculation: ₹998 (Mouse) + ₹99 (Notebook) = ₹1,097
    total_bill = sum(item["subtotal"] for item in cart)
    
    # Task: item_count reflects unique products (length of the list)
    return {
        "items": cart,
        "item_count": len(cart), 
        "grand_total": total_bill
    }

# **Question 3: Try Adding an Out-of-Stock Product (Validation)**
- Data validation is crucial for preventing inconsistent states in an application.
- In this scenario, we implement checks to ensure that:
- The requested product exists in the inventory (Handling 404 Not Found).
- The product is currently available in stock (Handling 400 Bad Request).
- By raising an HTTPException, we stop the execution of the function immediately and send a clear, helpful error message back to the client.

In [54]:
from fastapi import status

@app.post("/cart/add")
def add_to_cart(product_id: int, quantity: int = 1):
    # Task: Try POST /cart/add?product_id=99 — must return 404 Not Found
    if product_id not in inventory:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND, 
            detail="Product not found"
        )
    
    product = inventory[product_id]
    
    # Task: Try POST /cart/add?product_id=3 — USB Hub is out of stock
    # Must return 400 Bad Request with an out-of-stock error message
    if not product["in_stock"]:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST, 
            detail=f"{product['name']} is out of stock"
        )
    
    # Check for duplicates (Requirement from Q4)
    for item in cart:
        if item["product_id"] == product_id:
            item["quantity"] += quantity
            item["subtotal"] = item["unit_price"] * item["quantity"]
            return {"message": "Cart updated", "cart_item": item}

    # Add fresh item if validations pass
    new_item = {
        "product_id": product_id,
        "product_name": product["name"],
        "quantity": quantity,
        "unit_price": product["price"],
        "subtotal": product["price"] * quantity
    }
    cart.append(new_item)
    return {"message": "Added to cart", "cart_item": new_item}

# **Question 4: Add More Quantity of an Existing Cart Item**
- To provide a smooth user experience, a shopping cart should not create duplicate rows for the same product. Instead.
- it should detect if the item is already present and simply update the quantity and subtotal.
- This logic involves iterating through the current cart list and matching the product_id.

In [56]:
@app.post("/cart/add")
def add_to_cart(product_id: int, quantity: int = 1):
    if product_id not in inventory:
        raise HTTPException(status_code=404, detail="Product not found")
    
    product = inventory[product_id]
    
    if not product["in_stock"]:
        raise HTTPException(status_code=400, detail=f"{product['name']} is out of stock")

    # LOGIC FOR Q4: Check if product is already in the cart
    for item in cart:
        if item["product_id"] == product_id:
            item["quantity"] += quantity
            item["subtotal"] = item["unit_price"] * item["quantity"]
            return {"message": "Cart updated", "cart_item": item}

    # If product is NOT in cart, append as new item
    new_item = {
        "product_id": product_id,
        "product_name": product["name"],
        "quantity": quantity,
        "unit_price": product["price"],
        "subtotal": product["price"] * quantity
    }
    cart.append(new_item)
    return {"message": "Added to cart", "cart_item": new_item}

# **Question 5: Remove an Item Then Checkout**
- The checkout process marks the transition from a temporary state (the cart) to a permanent record (the orders).
- In this stage:
- Users can remove items they no longer want using the DELETE method.
- Upon checkout, the current cart contents are saved into an orders list along with customer metadata (name and address).
- The cart is then cleared, ensuring it is empty for the next session or purchase.

In [58]:
class CheckoutRequest(BaseModel):
    customer_name: str
    delivery_address: str

@app.delete("/cart/{product_id}")
def remove_from_cart(product_id: int):
    global cart
    initial_count = len(cart)
    # Rebuild cart list excluding the specific product_id
    cart = [item for item in cart if item["product_id"] != product_id]
    
    if len(cart) == initial_count:
        raise HTTPException(status_code=404, detail="Item not in cart")
    
    return {"message": "Item removed from cart"}

@app.post("/cart/checkout")
def checkout(request: CheckoutRequest):
    # Check if cart is empty (Bonus Requirement)
    if not cart:
        raise HTTPException(status_code=400, detail="Cart is empty — add items first")
    
    grand_total = sum(item["subtotal"] for item in cart)
    
    # Create the final order structure
    order_entry = {
        "order_id": len(orders) + 1,
        "customer_name": request.customer_name,
        "delivery_address": request.delivery_address,
        "items": list(cart), # Snapshot of current cart
        "total_amount": grand_total
    }
    
    orders.append(order_entry)
    cart.clear() # Resetting the cart
    
    return {"message": "Order placed successfully", "order": order_entry}

# **Question 6: Full Cart System Flow (Global History)**
- The final step is to verify the entire system's flow by tracking all successful checkouts.
- The GET /orders endpoint allows administrators or users to view the history of all transactions.
- This helps verify that multiple independent sessions (different customers) are correctly recorded in the permanent order history.

In [60]:
@app.get("/orders")
def get_all_orders():
    return {
        "total_orders": len(orders),
        "orders_history": orders
    }

# **Server Running Code**

In [74]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8004)

# Start server in a separate thread
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

INFO:     Started server process [15956]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8004 (Press CTRL+C to quit)
